# Bronze to Silver – Dynamic Tables (All 13 Tables)
*Co-authored with CoCo*

**Architecture:** Each bronze table gets a silver Dynamic Table that:
- **Deduplicates** via `QUALIFY ROW_NUMBER()` (no CTEs needed)
- **Normalises** text (TRIM, UPPER on flags/codes, LOWER on emails)
- **DQ checks** every record → `DQ_STATUS` (PASS/FAIL) + `DQ_FAIL_REASON`
- **Audits** lineage with `BRONZE_SOURCE_FILE`, `BRONZE_LOAD_TS`, `SILVER_LOAD_TS`

**Cost-saving choices:**
| Setting | Value | Why |
|---------|-------|-----|
| `TARGET_LAG` | `DOWNSTREAM` | Refreshes only when Gold layer DTs request it |
| `REFRESH_MODE` | `AUTO` | Snowflake picks INCREMENTAL when possible, FULL fallback |
| Stream on bronze? | **No** | DTs have built-in change tracking; Streams are for Tasks (imperative), not DTs (declarative) |

## 1. Region Master
Natural key: `REGION_CODE` &nbsp;|&nbsp; 4 rows

In [ ]:
%%sql -r region_ct
ALTER TABLE SALES_DEV.BRONZE.REGION_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r region_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.REGION_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked region master. Refreshes on downstream demand.'
AS
SELECT
    TRIM(REGION_CODE)              AS REGION_CODE,
    TRIM(REGION_NAME)              AS REGION_NAME,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(REGION_CODE)) = 0                            THEN 'FAIL'
        WHEN REGION_NAME IS NULL OR LENGTH(TRIM(REGION_NAME)) = 0     THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                THEN 'FAIL'
        WHEN EFFECTIVE_START_DATE IS NULL                              THEN 'FAIL'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE             THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(REGION_CODE)) = 0
            THEN 'REGION_CODE is blank after trim'
        WHEN REGION_NAME IS NULL OR LENGTH(TRIM(REGION_NAME)) = 0
            THEN 'REGION_NAME is null or blank'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')
            THEN 'IS_ACTIVE invalid – expected Y/N, got: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN EFFECTIVE_START_DATE IS NULL
            THEN 'EFFECTIVE_START_DATE is null'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE
            THEN 'EFFECTIVE_END_DATE (' || EFFECTIVE_END_DATE::VARCHAR
                 || ') precedes EFFECTIVE_START_DATE (' || EFFECTIVE_START_DATE::VARCHAR || ')'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME                    AS BRONZE_SOURCE_FILE,
    __LOAD_TS                      AS BRONZE_LOAD_TS,
    CURRENT_TIMESTAMP()            AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.REGION_MASTER
WHERE REGION_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY REGION_CODE
    ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC
) = 1;

In [ ]:
%%sql -r region_verify
SELECT * FROM SALES_DEV.SILVER.REGION_MASTER ORDER BY REGION_CODE;

## 2. Country Master
Natural key: `COUNTRY_CODE` &nbsp;|&nbsp; FKs: `REGION_CODE`, `CURRENCY_CODE`, `TAX_CODE` &nbsp;|&nbsp; 5 rows

In [ ]:
%%sql -r country_ct
ALTER TABLE SALES_DEV.BRONZE.COUNTRY_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r country_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.COUNTRY_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked country master. Refreshes on downstream demand.'
AS
SELECT
    TRIM(COUNTRY_CODE)                       AS COUNTRY_CODE,
    TRIM(COUNTRY_NAME)                       AS COUNTRY_NAME,
    TRIM(REGION_CODE)                        AS REGION_CODE,
    TRIM(CURRENCY_CODE)                      AS CURRENCY_CODE,
    TRIM(TAX_CODE)                           AS TAX_CODE,
    TRIM(PRIMARY_LANGUAGE)                   AS PRIMARY_LANGUAGE,
    TRIM(TIMEZONE)                           AS TIMEZONE,
    UPPER(TRIM(ECOMMERCE_SUPPORTED))         AS ECOMMERCE_SUPPORTED,
    UPPER(TRIM(RETAIL_STORE_SUPPORTED))      AS RETAIL_STORE_SUPPORTED,
    TRIM(MARKET_TIER)                        AS MARKET_TIER,
    CASE
        WHEN LENGTH(TRIM(COUNTRY_CODE)) = 0                               THEN 'FAIL'
        WHEN COUNTRY_NAME IS NULL OR LENGTH(TRIM(COUNTRY_NAME)) = 0       THEN 'FAIL'
        WHEN REGION_CODE IS NULL OR LENGTH(TRIM(REGION_CODE)) = 0         THEN 'FAIL'
        WHEN CURRENCY_CODE IS NULL OR LENGTH(TRIM(CURRENCY_CODE)) = 0     THEN 'FAIL'
        WHEN TAX_CODE IS NULL OR LENGTH(TRIM(TAX_CODE)) = 0               THEN 'FAIL'
        WHEN UPPER(TRIM(ECOMMERCE_SUPPORTED)) NOT IN ('Y', 'N')          THEN 'FAIL'
        WHEN UPPER(TRIM(RETAIL_STORE_SUPPORTED)) NOT IN ('Y', 'N')       THEN 'FAIL'
        ELSE 'PASS'
    END                                      AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(COUNTRY_CODE)) = 0
            THEN 'COUNTRY_CODE is blank after trim'
        WHEN COUNTRY_NAME IS NULL OR LENGTH(TRIM(COUNTRY_NAME)) = 0
            THEN 'COUNTRY_NAME is null or blank'
        WHEN REGION_CODE IS NULL OR LENGTH(TRIM(REGION_CODE)) = 0
            THEN 'REGION_CODE (FK) is null or blank'
        WHEN CURRENCY_CODE IS NULL OR LENGTH(TRIM(CURRENCY_CODE)) = 0
            THEN 'CURRENCY_CODE (FK) is null or blank'
        WHEN TAX_CODE IS NULL OR LENGTH(TRIM(TAX_CODE)) = 0
            THEN 'TAX_CODE (FK) is null or blank'
        WHEN UPPER(TRIM(ECOMMERCE_SUPPORTED)) NOT IN ('Y', 'N')
            THEN 'ECOMMERCE_SUPPORTED invalid – expected Y/N, got: ' || COALESCE(ECOMMERCE_SUPPORTED, 'NULL')
        WHEN UPPER(TRIM(RETAIL_STORE_SUPPORTED)) NOT IN ('Y', 'N')
            THEN 'RETAIL_STORE_SUPPORTED invalid – expected Y/N, got: ' || COALESCE(RETAIL_STORE_SUPPORTED, 'NULL')
        ELSE NULL
    END                                      AS DQ_FAIL_REASON,
    __FILE_NAME                              AS BRONZE_SOURCE_FILE,
    __LOAD_TS                                AS BRONZE_LOAD_TS,
    CURRENT_TIMESTAMP()                      AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.COUNTRY_MASTER
WHERE COUNTRY_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY COUNTRY_CODE
    ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC
) = 1;

In [ ]:
%%sql -r country_verify
SELECT * FROM SALES_DEV.SILVER.COUNTRY_MASTER ORDER BY COUNTRY_CODE;

## 3. Currency Master
Natural key: `CURRENCY_CODE` &nbsp;|&nbsp; 5 rows

In [ ]:
%%sql -r currency_ct
ALTER TABLE SALES_DEV.BRONZE.CURRENCY_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r currency_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.CURRENCY_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked currency master. Refreshes on downstream demand.'
AS
SELECT
    TRIM(CURRENCY_CODE)            AS CURRENCY_CODE,
    TRIM(CURRENCY_NAME)            AS CURRENCY_NAME,
    TRIM(CURRENCY_SYMBOL)          AS CURRENCY_SYMBOL,
    MINOR_UNIT,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(CURRENCY_CODE)) = 0                              THEN 'FAIL'
        WHEN CURRENCY_NAME IS NULL OR LENGTH(TRIM(CURRENCY_NAME)) = 0     THEN 'FAIL'
        WHEN CURRENCY_SYMBOL IS NULL OR LENGTH(TRIM(CURRENCY_SYMBOL)) = 0 THEN 'FAIL'
        WHEN MINOR_UNIT IS NULL OR MINOR_UNIT NOT BETWEEN 0 AND 4         THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                     THEN 'FAIL'
        WHEN EFFECTIVE_START_DATE IS NULL                                   THEN 'FAIL'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE                  THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(CURRENCY_CODE)) = 0
            THEN 'CURRENCY_CODE is blank after trim'
        WHEN CURRENCY_NAME IS NULL OR LENGTH(TRIM(CURRENCY_NAME)) = 0
            THEN 'CURRENCY_NAME is null or blank'
        WHEN CURRENCY_SYMBOL IS NULL OR LENGTH(TRIM(CURRENCY_SYMBOL)) = 0
            THEN 'CURRENCY_SYMBOL is null or blank'
        WHEN MINOR_UNIT IS NULL
            THEN 'MINOR_UNIT is null'
        WHEN MINOR_UNIT NOT BETWEEN 0 AND 4
            THEN 'MINOR_UNIT out of range (0-4): ' || MINOR_UNIT::VARCHAR
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')
            THEN 'IS_ACTIVE invalid – expected Y/N, got: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN EFFECTIVE_START_DATE IS NULL
            THEN 'EFFECTIVE_START_DATE is null'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE
            THEN 'EFFECTIVE_END_DATE (' || EFFECTIVE_END_DATE::VARCHAR
                 || ') precedes EFFECTIVE_START_DATE (' || EFFECTIVE_START_DATE::VARCHAR || ')'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME                    AS BRONZE_SOURCE_FILE,
    __LOAD_TS                      AS BRONZE_LOAD_TS,
    CURRENT_TIMESTAMP()            AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.CURRENCY_MASTER
WHERE CURRENCY_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY CURRENCY_CODE
    ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC
) = 1;

In [ ]:
%%sql -r currency_verify
SELECT * FROM SALES_DEV.SILVER.CURRENCY_MASTER ORDER BY CURRENCY_CODE;

## 4. Tax Master
Natural key: `TAX_CODE` &nbsp;|&nbsp; 5 rows

In [ ]:
%%sql -r tax_ct
ALTER TABLE SALES_DEV.BRONZE.TAX_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r tax_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.TAX_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked tax master. Refreshes on downstream demand.'
AS
SELECT
    TRIM(TAX_CODE)                 AS TAX_CODE,
    UPPER(TRIM(TAX_TYPE))          AS TAX_TYPE,
    TAX_RATE,
    UPPER(TRIM(TAX_INCLUSIVE_FLAG)) AS TAX_INCLUSIVE_FLAG,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(TAX_CODE)) = 0                               THEN 'FAIL'
        WHEN TAX_TYPE IS NULL OR LENGTH(TRIM(TAX_TYPE)) = 0           THEN 'FAIL'
        WHEN TAX_RATE IS NULL OR TAX_RATE < 0 OR TAX_RATE > 1        THEN 'FAIL'
        WHEN UPPER(TRIM(TAX_INCLUSIVE_FLAG)) NOT IN ('Y', 'N')        THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                THEN 'FAIL'
        WHEN EFFECTIVE_START_DATE IS NULL                              THEN 'FAIL'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE             THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(TAX_CODE)) = 0
            THEN 'TAX_CODE is blank after trim'
        WHEN TAX_TYPE IS NULL OR LENGTH(TRIM(TAX_TYPE)) = 0
            THEN 'TAX_TYPE is null or blank'
        WHEN TAX_RATE IS NULL
            THEN 'TAX_RATE is null'
        WHEN TAX_RATE < 0 OR TAX_RATE > 1
            THEN 'TAX_RATE out of range (0-1): ' || TAX_RATE::VARCHAR
        WHEN UPPER(TRIM(TAX_INCLUSIVE_FLAG)) NOT IN ('Y', 'N')
            THEN 'TAX_INCLUSIVE_FLAG invalid – expected Y/N, got: ' || COALESCE(TAX_INCLUSIVE_FLAG, 'NULL')
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')
            THEN 'IS_ACTIVE invalid – expected Y/N, got: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN EFFECTIVE_START_DATE IS NULL
            THEN 'EFFECTIVE_START_DATE is null'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE
            THEN 'EFFECTIVE_END_DATE (' || EFFECTIVE_END_DATE::VARCHAR
                 || ') precedes EFFECTIVE_START_DATE (' || EFFECTIVE_START_DATE::VARCHAR || ')'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME                    AS BRONZE_SOURCE_FILE,
    __LOAD_TS                      AS BRONZE_LOAD_TS,
    CURRENT_TIMESTAMP()            AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.TAX_MASTER
WHERE TAX_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY TAX_CODE
    ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC
) = 1;

In [ ]:
%%sql -r tax_verify
SELECT * FROM SALES_DEV.SILVER.TAX_MASTER ORDER BY TAX_CODE;

---
## 5. Product Category Master
Natural key: `CATEGORY_CODE` &nbsp;|&nbsp; 9 rows

In [ ]:
%%sql -r prod_cat_ct
ALTER TABLE SALES_DEV.BRONZE.PRODUCT_CATEGORY_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r prod_cat_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.PRODUCT_CATEGORY_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked product category master.'
AS
SELECT
    TRIM(CATEGORY_CODE)            AS CATEGORY_CODE,
    TRIM(CATEGORY_NAME)            AS CATEGORY_NAME,
    UPPER(TRIM(REPORTING_SEGMENT)) AS REPORTING_SEGMENT,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(CATEGORY_CODE)) = 0                                     THEN 'FAIL'
        WHEN CATEGORY_NAME IS NULL OR LENGTH(TRIM(CATEGORY_NAME)) = 0            THEN 'FAIL'
        WHEN REPORTING_SEGMENT IS NULL OR LENGTH(TRIM(REPORTING_SEGMENT)) = 0    THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                           THEN 'FAIL'
        WHEN EFFECTIVE_START_DATE IS NULL                                         THEN 'FAIL'
        WHEN EFFECTIVE_END_DATE IS NOT NULL
             AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE                        THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(CATEGORY_CODE)) = 0 THEN 'CATEGORY_CODE is blank'
        WHEN CATEGORY_NAME IS NULL OR LENGTH(TRIM(CATEGORY_NAME)) = 0 THEN 'CATEGORY_NAME is null or blank'
        WHEN REPORTING_SEGMENT IS NULL OR LENGTH(TRIM(REPORTING_SEGMENT)) = 0 THEN 'REPORTING_SEGMENT is null or blank'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N') THEN 'IS_ACTIVE invalid: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN EFFECTIVE_START_DATE IS NULL THEN 'EFFECTIVE_START_DATE is null'
        WHEN EFFECTIVE_END_DATE IS NOT NULL AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE THEN 'EFFECTIVE_END_DATE < EFFECTIVE_START_DATE'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.PRODUCT_CATEGORY_MASTER
WHERE CATEGORY_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY CATEGORY_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r prod_cat_verify
SELECT * FROM SALES_DEV.SILVER.PRODUCT_CATEGORY_MASTER ORDER BY CATEGORY_CODE;

## 6. Product Family Master
Natural key: `FAMILY_CODE` &nbsp;|&nbsp; FK: `CATEGORY_CODE` &nbsp;|&nbsp; 9 rows

In [ ]:
%%sql -r prod_fam_ct
ALTER TABLE SALES_DEV.BRONZE.PRODUCT_FAMILY_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r prod_fam_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.PRODUCT_FAMILY_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked product family master.'
AS
SELECT
    TRIM(FAMILY_CODE)              AS FAMILY_CODE,
    TRIM(FAMILY_NAME)              AS FAMILY_NAME,
    TRIM(CATEGORY_CODE)            AS CATEGORY_CODE,
    LAUNCH_YEAR,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    UPPER(TRIM(LIFECYCLE_STATUS))  AS LIFECYCLE_STATUS,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(FAMILY_CODE)) = 0                                       THEN 'FAIL'
        WHEN FAMILY_NAME IS NULL OR LENGTH(TRIM(FAMILY_NAME)) = 0                THEN 'FAIL'
        WHEN CATEGORY_CODE IS NULL OR LENGTH(TRIM(CATEGORY_CODE)) = 0            THEN 'FAIL'
        WHEN LAUNCH_YEAR IS NULL OR LAUNCH_YEAR < 1900 OR LAUNCH_YEAR > YEAR(CURRENT_DATE()) + 1 THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                           THEN 'FAIL'
        WHEN LIFECYCLE_STATUS IS NULL OR LENGTH(TRIM(LIFECYCLE_STATUS)) = 0      THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(FAMILY_CODE)) = 0 THEN 'FAMILY_CODE is blank'
        WHEN FAMILY_NAME IS NULL OR LENGTH(TRIM(FAMILY_NAME)) = 0 THEN 'FAMILY_NAME is null or blank'
        WHEN CATEGORY_CODE IS NULL OR LENGTH(TRIM(CATEGORY_CODE)) = 0 THEN 'CATEGORY_CODE (FK) is null or blank'
        WHEN LAUNCH_YEAR IS NULL THEN 'LAUNCH_YEAR is null'
        WHEN LAUNCH_YEAR < 1900 OR LAUNCH_YEAR > YEAR(CURRENT_DATE()) + 1 THEN 'LAUNCH_YEAR out of range: ' || LAUNCH_YEAR::VARCHAR
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N') THEN 'IS_ACTIVE invalid: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN LIFECYCLE_STATUS IS NULL OR LENGTH(TRIM(LIFECYCLE_STATUS)) = 0 THEN 'LIFECYCLE_STATUS is null or blank'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.PRODUCT_FAMILY_MASTER
WHERE FAMILY_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY FAMILY_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r prod_fam_verify
SELECT * FROM SALES_DEV.SILVER.PRODUCT_FAMILY_MASTER ORDER BY FAMILY_CODE;

## 7. Product Model Master
Natural key: `MODEL_CODE` &nbsp;|&nbsp; FK: `FAMILY_CODE` &nbsp;|&nbsp; 9 rows

In [ ]:
%%sql -r prod_model_ct
ALTER TABLE SALES_DEV.BRONZE.PRODUCT_MODEL_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r prod_model_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.PRODUCT_MODEL_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked product model master.'
AS
SELECT
    TRIM(MODEL_CODE)               AS MODEL_CODE,
    TRIM(MODEL_NAME)               AS MODEL_NAME,
    TRIM(FAMILY_CODE)              AS FAMILY_CODE,
    LAUNCH_DATE, DISCONTINUE_DATE,
    UPPER(TRIM(LIFECYCLE_STATUS))  AS LIFECYCLE_STATUS,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(MODEL_CODE)) = 0                                        THEN 'FAIL'
        WHEN MODEL_NAME IS NULL OR LENGTH(TRIM(MODEL_NAME)) = 0                  THEN 'FAIL'
        WHEN FAMILY_CODE IS NULL OR LENGTH(TRIM(FAMILY_CODE)) = 0                THEN 'FAIL'
        WHEN LAUNCH_DATE IS NULL                                                  THEN 'FAIL'
        WHEN DISCONTINUE_DATE IS NOT NULL AND DISCONTINUE_DATE < LAUNCH_DATE     THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                           THEN 'FAIL'
        WHEN LIFECYCLE_STATUS IS NULL OR LENGTH(TRIM(LIFECYCLE_STATUS)) = 0      THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(MODEL_CODE)) = 0 THEN 'MODEL_CODE is blank'
        WHEN MODEL_NAME IS NULL OR LENGTH(TRIM(MODEL_NAME)) = 0 THEN 'MODEL_NAME is null or blank'
        WHEN FAMILY_CODE IS NULL OR LENGTH(TRIM(FAMILY_CODE)) = 0 THEN 'FAMILY_CODE (FK) is null or blank'
        WHEN LAUNCH_DATE IS NULL THEN 'LAUNCH_DATE is null'
        WHEN DISCONTINUE_DATE IS NOT NULL AND DISCONTINUE_DATE < LAUNCH_DATE
            THEN 'DISCONTINUE_DATE (' || DISCONTINUE_DATE::VARCHAR || ') < LAUNCH_DATE (' || LAUNCH_DATE::VARCHAR || ')'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N') THEN 'IS_ACTIVE invalid: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN LIFECYCLE_STATUS IS NULL OR LENGTH(TRIM(LIFECYCLE_STATUS)) = 0 THEN 'LIFECYCLE_STATUS is null or blank'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.PRODUCT_MODEL_MASTER
WHERE MODEL_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY MODEL_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r prod_model_verify
SELECT * FROM SALES_DEV.SILVER.PRODUCT_MODEL_MASTER ORDER BY MODEL_CODE;

## 8. Product SKU Master
Natural key: `SKU_CODE` &nbsp;|&nbsp; FK: `MODEL_CODE` &nbsp;|&nbsp; 53 rows

In [ ]:
%%sql -r prod_sku_ct
ALTER TABLE SALES_DEV.BRONZE.PRODUCT_SKU_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r prod_sku_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.PRODUCT_SKU_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked product SKU master.'
AS
SELECT
    TRIM(SKU_CODE)                 AS SKU_CODE,
    TRIM(MODEL_CODE)               AS MODEL_CODE,
    TRIM(VARIANT)                  AS VARIANT,
    UPPER(TRIM(PRICE_TIER))        AS PRICE_TIER,
    GLOBAL_LAUNCH_DATE,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(SKU_CODE)) = 0                                          THEN 'FAIL'
        WHEN MODEL_CODE IS NULL OR LENGTH(TRIM(MODEL_CODE)) = 0                  THEN 'FAIL'
        WHEN VARIANT IS NULL OR LENGTH(TRIM(VARIANT)) = 0                        THEN 'FAIL'
        WHEN PRICE_TIER IS NULL OR LENGTH(TRIM(PRICE_TIER)) = 0                  THEN 'FAIL'
        WHEN GLOBAL_LAUNCH_DATE IS NULL                                           THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                           THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(SKU_CODE)) = 0 THEN 'SKU_CODE is blank'
        WHEN MODEL_CODE IS NULL OR LENGTH(TRIM(MODEL_CODE)) = 0 THEN 'MODEL_CODE (FK) is null or blank'
        WHEN VARIANT IS NULL OR LENGTH(TRIM(VARIANT)) = 0 THEN 'VARIANT is null or blank'
        WHEN PRICE_TIER IS NULL OR LENGTH(TRIM(PRICE_TIER)) = 0 THEN 'PRICE_TIER is null or blank'
        WHEN GLOBAL_LAUNCH_DATE IS NULL THEN 'GLOBAL_LAUNCH_DATE is null'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N') THEN 'IS_ACTIVE invalid: ' || COALESCE(IS_ACTIVE, 'NULL')
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.PRODUCT_SKU_MASTER
WHERE SKU_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY SKU_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r prod_sku_verify
SELECT * FROM SALES_DEV.SILVER.PRODUCT_SKU_MASTER ORDER BY SKU_CODE;

## 9. Product Country Availability
Composite key: `SKU_CODE` + `COUNTRY_CODE` &nbsp;|&nbsp; 371 rows

In [ ]:
%%sql -r prod_avail_ct
ALTER TABLE SALES_DEV.BRONZE.PRODUCT_COUNTRY_AVAILABILITY SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r prod_avail_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.PRODUCT_COUNTRY_AVAILABILITY
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked product-country availability.'
AS
SELECT
    TRIM(SKU_CODE)                 AS SKU_CODE,
    TRIM(COUNTRY_CODE)             AS COUNTRY_CODE,
    LOCAL_LAUNCH_DATE, LOCAL_DISCONTINUE_DATE,
    UPPER(TRIM(IS_AVAILABLE))      AS IS_AVAILABLE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(SKU_CODE)) = 0                                          THEN 'FAIL'
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0              THEN 'FAIL'
        WHEN LOCAL_LAUNCH_DATE IS NULL                                            THEN 'FAIL'
        WHEN LOCAL_DISCONTINUE_DATE IS NOT NULL
             AND LOCAL_DISCONTINUE_DATE < LOCAL_LAUNCH_DATE                       THEN 'FAIL'
        WHEN UPPER(TRIM(IS_AVAILABLE)) NOT IN ('Y', 'N')                         THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(SKU_CODE)) = 0 THEN 'SKU_CODE (FK) is blank'
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0 THEN 'COUNTRY_CODE (FK) is null or blank'
        WHEN LOCAL_LAUNCH_DATE IS NULL THEN 'LOCAL_LAUNCH_DATE is null'
        WHEN LOCAL_DISCONTINUE_DATE IS NOT NULL AND LOCAL_DISCONTINUE_DATE < LOCAL_LAUNCH_DATE
            THEN 'LOCAL_DISCONTINUE_DATE (' || LOCAL_DISCONTINUE_DATE::VARCHAR || ') < LOCAL_LAUNCH_DATE (' || LOCAL_LAUNCH_DATE::VARCHAR || ')'
        WHEN UPPER(TRIM(IS_AVAILABLE)) NOT IN ('Y', 'N') THEN 'IS_AVAILABLE invalid: ' || COALESCE(IS_AVAILABLE, 'NULL')
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.PRODUCT_COUNTRY_AVAILABILITY
WHERE SKU_CODE IS NOT NULL AND COUNTRY_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY SKU_CODE, COUNTRY_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r prod_avail_verify
SELECT * FROM SALES_DEV.SILVER.PRODUCT_COUNTRY_AVAILABILITY ORDER BY SKU_CODE, COUNTRY_CODE;

---
## 10. Store Master
Natural key: `STORE_CODE` &nbsp;|&nbsp; FKs: `COUNTRY_CODE`, `REGION_CODE` &nbsp;|&nbsp; 80 rows

Domain checks: geo coordinates, floor area > 0, rent >= 0, close >= open, effective date range

In [ ]:
%%sql -r store_ct
ALTER TABLE SALES_DEV.BRONZE.STORE_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r store_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.STORE_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked store master.'
AS
SELECT
    TRIM(STORE_CODE)               AS STORE_CODE,
    TRIM(STORE_NAME)               AS STORE_NAME,
    TRIM(COUNTRY_CODE)             AS COUNTRY_CODE,
    TRIM(REGION_CODE)              AS REGION_CODE,
    TRIM(TAX_JURISDICTION_CODE)    AS TAX_JURISDICTION_CODE,
    UPPER(TRIM(FORMAT_CODE))       AS FORMAT_CODE,
    TRIM(CITY)                     AS CITY,
    TRIM(STATE_CODE)               AS STATE_CODE,
    TRIM(POSTAL_CODE)              AS POSTAL_CODE,
    TRIM(ADDRESS_LINE1)            AS ADDRESS_LINE1,
    LATITUDE, LONGITUDE,
    STORE_OPEN_DATE, STORE_CLOSE_DATE,
    UPPER(TRIM(LIFECYCLE_STATUS))  AS LIFECYCLE_STATUS,
    FLOOR_AREA_SQFT, ANNUAL_RENT_USD,
    UPPER(TRIM(IS_ACTIVE))         AS IS_ACTIVE,
    EFFECTIVE_START_DATE, EFFECTIVE_END_DATE,
    CREATED_AT,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    CASE
        WHEN LENGTH(TRIM(STORE_CODE)) = 0                                        THEN 'FAIL'
        WHEN STORE_NAME IS NULL OR LENGTH(TRIM(STORE_NAME)) = 0                  THEN 'FAIL'
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0              THEN 'FAIL'
        WHEN REGION_CODE IS NULL OR LENGTH(TRIM(REGION_CODE)) = 0                THEN 'FAIL'
        WHEN FORMAT_CODE IS NULL OR LENGTH(TRIM(FORMAT_CODE)) = 0                THEN 'FAIL'
        WHEN STORE_OPEN_DATE IS NULL                                              THEN 'FAIL'
        WHEN STORE_CLOSE_DATE IS NOT NULL AND STORE_CLOSE_DATE < STORE_OPEN_DATE THEN 'FAIL'
        WHEN LATITUDE IS NOT NULL AND (LATITUDE < -90 OR LATITUDE > 90)          THEN 'FAIL'
        WHEN LONGITUDE IS NOT NULL AND (LONGITUDE < -180 OR LONGITUDE > 180)     THEN 'FAIL'
        WHEN FLOOR_AREA_SQFT IS NOT NULL AND FLOOR_AREA_SQFT <= 0               THEN 'FAIL'
        WHEN ANNUAL_RENT_USD IS NOT NULL AND ANNUAL_RENT_USD < 0                 THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N')                           THEN 'FAIL'
        WHEN EFFECTIVE_START_DATE IS NULL                                         THEN 'FAIL'
        WHEN EFFECTIVE_END_DATE IS NOT NULL AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(STORE_CODE)) = 0 THEN 'STORE_CODE is blank'
        WHEN STORE_NAME IS NULL OR LENGTH(TRIM(STORE_NAME)) = 0 THEN 'STORE_NAME is null or blank'
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0 THEN 'COUNTRY_CODE (FK) is null or blank'
        WHEN REGION_CODE IS NULL OR LENGTH(TRIM(REGION_CODE)) = 0 THEN 'REGION_CODE (FK) is null or blank'
        WHEN FORMAT_CODE IS NULL OR LENGTH(TRIM(FORMAT_CODE)) = 0 THEN 'FORMAT_CODE is null or blank'
        WHEN STORE_OPEN_DATE IS NULL THEN 'STORE_OPEN_DATE is null'
        WHEN STORE_CLOSE_DATE IS NOT NULL AND STORE_CLOSE_DATE < STORE_OPEN_DATE
            THEN 'STORE_CLOSE_DATE (' || STORE_CLOSE_DATE::VARCHAR || ') < STORE_OPEN_DATE (' || STORE_OPEN_DATE::VARCHAR || ')'
        WHEN LATITUDE IS NOT NULL AND (LATITUDE < -90 OR LATITUDE > 90) THEN 'LATITUDE out of range: ' || LATITUDE::VARCHAR
        WHEN LONGITUDE IS NOT NULL AND (LONGITUDE < -180 OR LONGITUDE > 180) THEN 'LONGITUDE out of range: ' || LONGITUDE::VARCHAR
        WHEN FLOOR_AREA_SQFT IS NOT NULL AND FLOOR_AREA_SQFT <= 0 THEN 'FLOOR_AREA_SQFT must be positive: ' || FLOOR_AREA_SQFT::VARCHAR
        WHEN ANNUAL_RENT_USD IS NOT NULL AND ANNUAL_RENT_USD < 0 THEN 'ANNUAL_RENT_USD is negative: ' || ANNUAL_RENT_USD::VARCHAR
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('Y', 'N') THEN 'IS_ACTIVE invalid: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN EFFECTIVE_START_DATE IS NULL THEN 'EFFECTIVE_START_DATE is null'
        WHEN EFFECTIVE_END_DATE IS NOT NULL AND EFFECTIVE_END_DATE < EFFECTIVE_START_DATE THEN 'EFFECTIVE_END_DATE < EFFECTIVE_START_DATE'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.STORE_MASTER
WHERE STORE_CODE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY STORE_CODE ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r store_verify
SELECT * FROM SALES_DEV.SILVER.STORE_MASTER ORDER BY STORE_CODE;

## 11. Customer Master
Natural key: `CUSTOMER_ID` (UUID) &nbsp;|&nbsp; FKs: `COUNTRY_CODE`, `REGION` &nbsp;|&nbsp; 50,000 rows

`IS_ACTIVE` normalised from `True/False` → `Y/N`; `EMAIL` lowercased

In [ ]:
%%sql -r cust_ct
ALTER TABLE SALES_DEV.BRONZE.CUSTOMER_MASTER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r cust_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.CUSTOMER_MASTER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked customer master.'
AS
SELECT
    TRIM(CUSTOMER_ID)              AS CUSTOMER_ID,
    TRIM(CUSTOMER_NUMBER)          AS CUSTOMER_NUMBER,
    TRIM(FIRST_NAME)               AS FIRST_NAME,
    TRIM(LAST_NAME)                AS LAST_NAME,
    TRIM(FULL_NAME)                AS FULL_NAME,
    UPPER(TRIM(GENDER))            AS GENDER,
    DATE_OF_BIRTH,
    LOWER(TRIM(EMAIL))             AS EMAIL,
    TRIM(PHONE_NUMBER)             AS PHONE_NUMBER,
    TRIM(STREET_ADDRESS)           AS STREET_ADDRESS,
    TRIM(CITY)                     AS CITY,
    TRIM(STATE_PROVINCE)           AS STATE_PROVINCE,
    TRIM(POSTAL_CODE)              AS POSTAL_CODE,
    TRIM(COUNTRY_CODE)             AS COUNTRY_CODE,
    TRIM(COUNTRY_NAME)             AS COUNTRY_NAME,
    TRIM(REGION)                   AS REGION,
    TRIM(PREFERRED_LANGUAGE)       AS PREFERRED_LANGUAGE,
    UPPER(TRIM(CUSTOMER_SEGMENT))  AS CUSTOMER_SEGMENT,
    UPPER(TRIM(LOYALTY_TIER))      AS LOYALTY_TIER,
    REGISTRATION_DATE,
    CASE UPPER(TRIM(IS_ACTIVE))
        WHEN 'TRUE'  THEN 'Y'
        WHEN 'FALSE' THEN 'N'
        ELSE UPPER(TRIM(IS_ACTIVE))
    END                            AS IS_ACTIVE,
    TRIM(SOURCE_SYSTEM)            AS SOURCE_SYSTEM,
    UPPER(TRIM(RECORD_SOURCE))     AS RECORD_SOURCE,
    CREATED_AT, UPDATED_AT,
    CASE
        WHEN LENGTH(TRIM(CUSTOMER_ID)) = 0                                       THEN 'FAIL'
        WHEN CUSTOMER_NUMBER IS NULL OR LENGTH(TRIM(CUSTOMER_NUMBER)) = 0        THEN 'FAIL'
        WHEN FULL_NAME IS NULL OR LENGTH(TRIM(FULL_NAME)) = 0                    THEN 'FAIL'
        WHEN EMAIL IS NULL OR LENGTH(TRIM(EMAIL)) = 0                            THEN 'FAIL'
        WHEN EMAIL IS NOT NULL AND EMAIL NOT LIKE '%@%.%'                        THEN 'FAIL'
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0              THEN 'FAIL'
        WHEN REGISTRATION_DATE IS NULL                                            THEN 'FAIL'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('TRUE', 'FALSE')                    THEN 'FAIL'
        WHEN CUSTOMER_SEGMENT IS NULL OR LENGTH(TRIM(CUSTOMER_SEGMENT)) = 0     THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(CUSTOMER_ID)) = 0 THEN 'CUSTOMER_ID is blank'
        WHEN CUSTOMER_NUMBER IS NULL OR LENGTH(TRIM(CUSTOMER_NUMBER)) = 0 THEN 'CUSTOMER_NUMBER is null or blank'
        WHEN FULL_NAME IS NULL OR LENGTH(TRIM(FULL_NAME)) = 0 THEN 'FULL_NAME is null or blank'
        WHEN EMAIL IS NULL OR LENGTH(TRIM(EMAIL)) = 0 THEN 'EMAIL is null or blank'
        WHEN EMAIL IS NOT NULL AND EMAIL NOT LIKE '%@%.%' THEN 'EMAIL format invalid: ' || EMAIL
        WHEN COUNTRY_CODE IS NULL OR LENGTH(TRIM(COUNTRY_CODE)) = 0 THEN 'COUNTRY_CODE (FK) is null or blank'
        WHEN REGISTRATION_DATE IS NULL THEN 'REGISTRATION_DATE is null'
        WHEN UPPER(TRIM(IS_ACTIVE)) NOT IN ('TRUE', 'FALSE')
            THEN 'IS_ACTIVE invalid – expected True/False, got: ' || COALESCE(IS_ACTIVE, 'NULL')
        WHEN CUSTOMER_SEGMENT IS NULL OR LENGTH(TRIM(CUSTOMER_SEGMENT)) = 0 THEN 'CUSTOMER_SEGMENT is null or blank'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.CUSTOMER_MASTER
WHERE CUSTOMER_ID IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r cust_verify
SELECT COUNT(*) AS ROW_COUNT, COUNT(CASE WHEN DQ_STATUS = 'FAIL' THEN 1 END) AS FAILED_ROWS
FROM SALES_DEV.SILVER.CUSTOMER_MASTER;

---
## 12. Sales Header
Natural key: `TRANSACTION_ID` (UUID) &nbsp;|&nbsp; FKs: `CUSTOMER_ID`, `STORE_ID`, `CURRENCY` &nbsp;|&nbsp; 86,107 rows

Arithmetic check: `gross - discount + tax ≈ net_total` (±0.02 tolerance)

In [ ]:
%%sql -r sales_hdr_ct
ALTER TABLE SALES_DEV.BRONZE.SALES_HEADER SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r sales_hdr_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.SALES_HEADER
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked sales transaction headers.'
AS
SELECT
    TRIM(TRANSACTION_ID)           AS TRANSACTION_ID,
    TRIM(TRANSACTION_NUMBER)       AS TRANSACTION_NUMBER,
    TRANSACTION_TIMESTAMP,
    TRIM(CUSTOMER_ID)              AS CUSTOMER_ID,
    TRIM(STORE_ID)                 AS STORE_ID,
    UPPER(TRIM(CHANNEL_ID))        AS CHANNEL_ID,
    TRIM(PAYMENT_METHOD)           AS PAYMENT_METHOD,
    UPPER(TRIM(CURRENCY))          AS CURRENCY,
    GROSS_AMOUNT, TOTAL_DISCOUNT, TOTAL_TAX, NET_TOTAL,
    CREATED_AT,
    CASE
        WHEN LENGTH(TRIM(TRANSACTION_ID)) = 0                                    THEN 'FAIL'
        WHEN TRANSACTION_NUMBER IS NULL OR LENGTH(TRIM(TRANSACTION_NUMBER)) = 0  THEN 'FAIL'
        WHEN TRANSACTION_TIMESTAMP IS NULL                                        THEN 'FAIL'
        WHEN CUSTOMER_ID IS NULL OR LENGTH(TRIM(CUSTOMER_ID)) = 0               THEN 'FAIL'
        WHEN STORE_ID IS NULL OR LENGTH(TRIM(STORE_ID)) = 0                     THEN 'FAIL'
        WHEN CHANNEL_ID IS NULL OR LENGTH(TRIM(CHANNEL_ID)) = 0                 THEN 'FAIL'
        WHEN CURRENCY IS NULL OR LENGTH(TRIM(CURRENCY)) = 0                     THEN 'FAIL'
        WHEN GROSS_AMOUNT IS NULL OR GROSS_AMOUNT < 0                            THEN 'FAIL'
        WHEN NET_TOTAL IS NULL OR NET_TOTAL < 0                                  THEN 'FAIL'
        WHEN TOTAL_DISCOUNT IS NOT NULL AND TOTAL_DISCOUNT < 0                   THEN 'FAIL'
        WHEN TOTAL_TAX IS NOT NULL AND TOTAL_TAX < 0                             THEN 'FAIL'
        WHEN ABS(GROSS_AMOUNT - TOTAL_DISCOUNT + TOTAL_TAX - NET_TOTAL) > 0.02  THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(TRANSACTION_ID)) = 0 THEN 'TRANSACTION_ID is blank'
        WHEN TRANSACTION_NUMBER IS NULL OR LENGTH(TRIM(TRANSACTION_NUMBER)) = 0 THEN 'TRANSACTION_NUMBER is null or blank'
        WHEN TRANSACTION_TIMESTAMP IS NULL THEN 'TRANSACTION_TIMESTAMP is null'
        WHEN CUSTOMER_ID IS NULL OR LENGTH(TRIM(CUSTOMER_ID)) = 0 THEN 'CUSTOMER_ID (FK) is null or blank'
        WHEN STORE_ID IS NULL OR LENGTH(TRIM(STORE_ID)) = 0 THEN 'STORE_ID (FK) is null or blank'
        WHEN CHANNEL_ID IS NULL OR LENGTH(TRIM(CHANNEL_ID)) = 0 THEN 'CHANNEL_ID is null or blank'
        WHEN CURRENCY IS NULL OR LENGTH(TRIM(CURRENCY)) = 0 THEN 'CURRENCY (FK) is null or blank'
        WHEN GROSS_AMOUNT IS NULL OR GROSS_AMOUNT < 0 THEN 'GROSS_AMOUNT is null or negative'
        WHEN NET_TOTAL IS NULL OR NET_TOTAL < 0 THEN 'NET_TOTAL is null or negative'
        WHEN TOTAL_DISCOUNT IS NOT NULL AND TOTAL_DISCOUNT < 0 THEN 'TOTAL_DISCOUNT is negative'
        WHEN TOTAL_TAX IS NOT NULL AND TOTAL_TAX < 0 THEN 'TOTAL_TAX is negative'
        WHEN ABS(GROSS_AMOUNT - TOTAL_DISCOUNT + TOTAL_TAX - NET_TOTAL) > 0.02
            THEN 'Amount mismatch: gross(' || GROSS_AMOUNT::VARCHAR || ') - disc(' || TOTAL_DISCOUNT::VARCHAR
                 || ') + tax(' || TOTAL_TAX::VARCHAR || ') != net(' || NET_TOTAL::VARCHAR || ')'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.SALES_HEADER
WHERE TRANSACTION_ID IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY TRANSACTION_ID ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r sales_hdr_verify
SELECT COUNT(*) AS ROW_COUNT, COUNT(CASE WHEN DQ_STATUS = 'FAIL' THEN 1 END) AS FAILED_ROWS
FROM SALES_DEV.SILVER.SALES_HEADER;

## 13. Sales Item (Line Items)
Natural key: `TRANSACTION_LINE_ID` (UUID) &nbsp;|&nbsp; FKs: `TRANSACTION_ID`, `SKU_CODE` &nbsp;|&nbsp; 86,107 rows

Arithmetic check: `qty × price - discount + tax ≈ line_total` (±0.02 tolerance)

In [ ]:
%%sql -r sales_item_ct
ALTER TABLE SALES_DEV.BRONZE.SALES_ITEM SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r sales_item_dt
CREATE OR REPLACE DYNAMIC TABLE SALES_DEV.SILVER.SALES_ITEM
    TARGET_LAG   = 'DOWNSTREAM'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    COMMENT = 'Silver – deduplicated, normalised, DQ-checked sales line items.'
AS
SELECT
    TRIM(TRANSACTION_LINE_ID)      AS TRANSACTION_LINE_ID,
    TRIM(TRANSACTION_ID)           AS TRANSACTION_ID,
    TRIM(SKU_CODE)                 AS SKU_CODE,
    QUANTITY, UNIT_PRICE, DISCOUNT_AMOUNT, TAX_AMOUNT, LINE_TOTAL,
    CREATED_AT,
    CASE
        WHEN LENGTH(TRIM(TRANSACTION_LINE_ID)) = 0                               THEN 'FAIL'
        WHEN TRANSACTION_ID IS NULL OR LENGTH(TRIM(TRANSACTION_ID)) = 0          THEN 'FAIL'
        WHEN SKU_CODE IS NULL OR LENGTH(TRIM(SKU_CODE)) = 0                      THEN 'FAIL'
        WHEN QUANTITY IS NULL OR QUANTITY <= 0                                     THEN 'FAIL'
        WHEN UNIT_PRICE IS NULL OR UNIT_PRICE < 0                                THEN 'FAIL'
        WHEN DISCOUNT_AMOUNT IS NOT NULL AND DISCOUNT_AMOUNT < 0                 THEN 'FAIL'
        WHEN TAX_AMOUNT IS NOT NULL AND TAX_AMOUNT < 0                           THEN 'FAIL'
        WHEN LINE_TOTAL IS NULL OR LINE_TOTAL < 0                                THEN 'FAIL'
        WHEN ABS((QUANTITY * UNIT_PRICE) - COALESCE(DISCOUNT_AMOUNT, 0)
                 + COALESCE(TAX_AMOUNT, 0) - LINE_TOTAL) > 0.02                  THEN 'FAIL'
        ELSE 'PASS'
    END                            AS DQ_STATUS,
    CASE
        WHEN LENGTH(TRIM(TRANSACTION_LINE_ID)) = 0 THEN 'TRANSACTION_LINE_ID is blank'
        WHEN TRANSACTION_ID IS NULL OR LENGTH(TRIM(TRANSACTION_ID)) = 0 THEN 'TRANSACTION_ID (FK) is null or blank'
        WHEN SKU_CODE IS NULL OR LENGTH(TRIM(SKU_CODE)) = 0 THEN 'SKU_CODE (FK) is null or blank'
        WHEN QUANTITY IS NULL OR QUANTITY <= 0 THEN 'QUANTITY is null or non-positive: ' || COALESCE(QUANTITY::VARCHAR, 'NULL')
        WHEN UNIT_PRICE IS NULL OR UNIT_PRICE < 0 THEN 'UNIT_PRICE is null or negative: ' || COALESCE(UNIT_PRICE::VARCHAR, 'NULL')
        WHEN DISCOUNT_AMOUNT IS NOT NULL AND DISCOUNT_AMOUNT < 0 THEN 'DISCOUNT_AMOUNT is negative: ' || DISCOUNT_AMOUNT::VARCHAR
        WHEN TAX_AMOUNT IS NOT NULL AND TAX_AMOUNT < 0 THEN 'TAX_AMOUNT is negative: ' || TAX_AMOUNT::VARCHAR
        WHEN LINE_TOTAL IS NULL OR LINE_TOTAL < 0 THEN 'LINE_TOTAL is null or negative'
        WHEN ABS((QUANTITY * UNIT_PRICE) - COALESCE(DISCOUNT_AMOUNT, 0)
                 + COALESCE(TAX_AMOUNT, 0) - LINE_TOTAL) > 0.02
            THEN 'Line total mismatch: qty(' || QUANTITY::VARCHAR || ') * price(' || UNIT_PRICE::VARCHAR
                 || ') - disc(' || COALESCE(DISCOUNT_AMOUNT, 0)::VARCHAR || ') + tax(' || COALESCE(TAX_AMOUNT, 0)::VARCHAR
                 || ') != line_total(' || LINE_TOTAL::VARCHAR || ')'
        ELSE NULL
    END                            AS DQ_FAIL_REASON,
    __FILE_NAME AS BRONZE_SOURCE_FILE, __LOAD_TS AS BRONZE_LOAD_TS, CURRENT_TIMESTAMP() AS SILVER_LOAD_TS
FROM SALES_DEV.BRONZE.SALES_ITEM
WHERE TRANSACTION_LINE_ID IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY TRANSACTION_LINE_ID ORDER BY __LOAD_TS DESC, __ROW_NUMBER ASC) = 1;

In [ ]:
%%sql -r sales_item_verify
SELECT COUNT(*) AS ROW_COUNT, COUNT(CASE WHEN DQ_STATUS = 'FAIL' THEN 1 END) AS FAILED_ROWS
FROM SALES_DEV.SILVER.SALES_ITEM;